# 阶段八：树——从入门到精通

> **学习目标**：建立"树形结构"的总体认知，掌握二叉树和 BST 的核心操作，理解树与森林的转换，并学习哈夫曼树的构造与编码。
> **预计学时**：约 5.6 小时

---

## 第一章 树的基本概念

### 1.1 什么是树？

在学习链表、栈、队列时，我们接触的都是**线性结构**——数据元素"一个接一个"排列。但现实中有大量的数据具有**层次关系**：

- 公司的组织架构（CEO → 部门经理 → 员工）
- 文件系统（根目录 → 文件夹 → 文件）
- HTML 文档的 DOM 结构

这些场景都无法用线性结构高效表示，我们需要一种新的数据结构——**树 (Tree)**。

#### 1.1.1 树的递归定义

> **树**是 $n$（$n \ge 0$）个结点的有限集合 $T$。当 $n = 0$ 时，称为**空树**；当 $n > 0$ 时，满足：
> 1. 有且仅有一个特定的结点称为**根 (Root)**；
> 2. 其余结点可分为 $m$（$m \ge 0$）个**互不相交**的有限集合 $T_1, T_2, \ldots, T_m$，其中每个集合本身又是一棵树，称为根的**子树 (Subtree)**。

这是一个**递归定义**——树的定义中包含了树自身。这个特征将贯穿整个树的学习过程：树的绝大多数算法，本质上都是递归的。

```
         A          ← 根结点
       / | \
      B  C  D       ← A 的三棵子树的根
     / \    |
    E   F   G       ← 叶子结点 E、G；F 有子结点 H
        |
        H           ← 叶子结点
```

**关键约束**：子树之间**互不相交**。如果两棵子树有交集，那就不是树，而是"图"了。

---

### 1.2 基本术语（必须烂熟于心）

| 术语                     | 定义                                               | 上图示例                              |
| ------------------------ | -------------------------------------------------- | ------------------------------------- |
| **结点 (Node)**          | 树中的一个数据元素                                 | A, B, C, D, E, F, G, H                |
| **根结点 (Root)**        | 没有前驱（父结点）的结点                           | A                                     |
| **叶子/终端结点 (Leaf)** | 没有后继（子结点）的结点                           | E, C, H, G                            |
| **分支/非终端结点**      | 有子结点的结点                                     | A, B, D, F                            |
| **边 (Edge)**            | 父子结点之间的连线                                 | A→B, B→E, ...                         |
| **子结点 (Child)**       | 某结点的直接后继                                   | B 是 A 的子结点                       |
| **父结点 (Parent)**      | 某结点的直接前驱                                   | A 是 B 的父结点                       |
| **兄弟 (Sibling)**       | 同一父结点的子结点                                 | B、C、D 互为兄弟                      |
| **祖先 (Ancestor)**      | 从根到该结点路径上的所有结点                       | H 的祖先：F、B、A                     |
| **子孙 (Descendant)**    | 以某结点为根的子树中的所有结点                     | A 的子孙：所有其他结点                |
| **结点的度 (Degree)**    | 该结点拥有的子树个数                               | degree(A)=3, degree(B)=2, degree(E)=0 |
| **树的度**               | 树中所有结点度的最大值                             | degree(T) = 3（取 A 的度）            |
| **层次 (Level)**         | 根在第 1 层，根的子结点在第 2 层...                | A→1层, B/C/D→2层, E/F/G→3层, H→4层    |
| **深度 (Depth)**         | 从根到该结点的路径长度（边数或层数，注意教材差异） | depth(H) = 3（边数）或 4（层数）      |
| **高度 (Height)**        | 从该结点到最远叶子的路径长度                       | height(A) = 3（边数）                 |
| **树的高度/深度**        | 根结点的高度（或最大层次）                         | 3（边数）或 4（层数）                 |
| **有序树**               | 子树之间有确定的左右次序                           | —                                     |
| **无序树**               | 子树之间没有次序                                   | —                                     |
| **森林 (Forest)**        | $m$（$m \ge 0$）棵互不相交的树的集合               | 删去根 A 后，剩余 3 棵子树构成森林    |

> ⚠️ **深度/高度的定义在不同教材中有差异**。王道考研中通常定义：根结点的深度为 1（按层次计）；而很多英文教材定义根结点的深度为 0（按边数计）。做题时务必确认教材约定。

---

### 1.3 树的重要性质（考试高频考点）

这些性质看似简单，但在选择题中反复出现，务必理解推导过程。

#### 性质 1：结点数 = 总度数 + 1

$$n = \sum_{i} degree(i) + 1$$

**推导**：每条边连接一对父子结点，而每个非根结点恰好有一个父结点（即恰好被一条边"指向"）。因此边数 = 非根结点数 = $n - 1$。另一方面，每个结点的度就是它"射出"的边数，所以边数 = 总度数。两式合并得：$n - 1 = \text{总度数}$，即 $n = \text{总度数} + 1$。

#### 性质 2：度为 $m$ 的树中，第 $i$ 层最多有 $m^{i-1}$ 个结点

**推导**：第 1 层只有根，1 个结点。每个结点最多 $m$ 个子结点，所以第 2 层最多 $m$ 个，第 3 层最多 $m^2$ 个……第 $i$ 层最多 $m^{i-1}$ 个。

#### 性质 3：高度为 $h$ 的 $m$ 叉树，最多有 $\dfrac{m^h - 1}{m - 1}$ 个结点（高度按层数计）

**推导**：这是一个等比数列求和。最多结点数 = $1 + m + m^2 + \cdots + m^{h-1} = \dfrac{m^h - 1}{m - 1}$。

#### 性质 4：具有 $n$ 个结点的 $m$ 叉树的最小高度（高度按层数计）

将上面的公式反过来：$n \le \dfrac{m^h - 1}{m - 1}$，解出 $h \ge \log_m[n(m-1) + 1]$。

$$h_{min} = \lceil \log_m[n(m-1) + 1] \rceil$$

---

## 第二章 二叉树 (Binary Tree)

### 2.1 为什么要单独研究二叉树？

二叉树是树的一种特殊形式，它有极其优美的数学性质和极其广泛的应用。更重要的是：

> **任意树、任意森林都可以转换为二叉树**来处理（后面会详细讲）。

所以，二叉树是整个"树"理论的核心基石。

### 2.2 二叉树的定义

> **二叉树**是 $n$（$n \ge 0$）个结点的有限集合，它或者是空树，或者由一个根结点和两棵互不相交的、分别称为**左子树**和**右子树**的二叉树组成。

**与度为 2 的树的区别**：

|                   | 二叉树                 | 度为 2 的有序树               |
| ----------------- | ---------------------- | ----------------------------- |
| 子树区分          | **严格区分左右子树**   | 仅在有 2 个子树时才有左右之分 |
| 允许为空          | 可以是空树             | 至少有 3 个结点               |
| 只有 1 个子结点时 | 必须明确是左子还是右子 | 无所谓左右                    |

这个区别非常重要，考试常考！

### 2.3 五种基本形态

```
(1) 空树    (2) 只有根    (3) 只有左子树   (4) 只有右子树   (5) 左右子树都有

   ∅          A             A              A              A
                           /                \            / \
                          B                  B          B   C
```

### 2.4 特殊的二叉树

#### 满二叉树 (Full/Perfect Binary Tree)

每一层的结点数都达到最大值。若高度按层数计，高度为 $h$ 的满二叉树有 $2^h - 1$ 个结点。

```
         1
       /   \
      2     3
     / \   / \
    4   5 6   7
```

**特点**：
- 叶子结点全部在最底层
- 不存在度为 1 的结点
- 按层序编号，结点 $i$ 的左子结点为 $2i$，右子结点为 $2i + 1$，父结点为 $\lfloor i/2 \rfloor$

#### 完全二叉树 (Complete Binary Tree)

设二叉树高度为 $h$，前 $h-1$ 层都是满的，第 $h$ 层的结点从左到右连续排列（中间不能有空缺）。

```
         1                1
       /   \            /   \
      2     3          2     3
     / \   /          / \
    4   5 6          4   5

    完全二叉树         完全二叉树
```

```
         1
       /   \
      2     3
     / \     \
    4   5     7

   ❌ 不是完全二叉树（第 3 层结点不连续，缺少 6）
```

**完全二叉树的重要性质**：

1. 具有 $n$ 个结点的完全二叉树的高度（按层数计）：$h = \lfloor \log_2 n \rfloor + 1$
2. 度为 1 的结点最多 1 个（且只可能有左子结点）
3. 叶子结点只出现在最后两层
4. 可以用**数组**顺序存储，不浪费空间（这就是**堆**的基础！）

### 2.5 二叉树的性质（重点记忆）

#### 性质 1：非空二叉树的第 $i$ 层最多 $2^{i-1}$ 个结点

（这是树的性质 2 在 $m=2$ 时的特例）

#### 性质 2：高度为 $h$ 的二叉树最多 $2^h - 1$ 个结点（高度按层数计）

$$1 + 2 + 4 + \cdots + 2^{h-1} = 2^h - 1$$

#### 性质 3（重要！）：$n_0 = n_2 + 1$

设度为 0（叶子）、1、2 的结点个数分别为 $n_0, n_1, n_2$，则：

$$n_0 = n_2 + 1$$

**推导**：

- 总结点数：$n = n_0 + n_1 + n_2$ ……①
- 总边数 = $n - 1$（每个非根结点头顶一条边）
- 总边数 = $n_1 \cdot 1 + n_2 \cdot 2$（度为 $k$ 的结点射出 $k$ 条边）
- 所以：$n - 1 = n_1 + 2n_2$ ……②
- ① - ②：$1 = n_0 - n_2$，即 $n_0 = n_2 + 1$

> 💡 **记忆口诀**：叶子比双分支多一个。

#### 性质 4：完全二叉树的顺序编号关系

对 $n$ 个结点的完全二叉树，按层序从 1 开始编号：

| 关系           | 公式                             |
| -------------- | -------------------------------- |
| $i$ 的父结点   | $\lfloor i/2 \rfloor$（$i > 1$） |
| $i$ 的左子结点 | $2i$（若 $2i \le n$）            |
| $i$ 的右子结点 | $2i + 1$（若 $2i + 1 \le n$）    |
| $i$ 是否为叶子 | $i > \lfloor n/2 \rfloor$        |
| $i$ 所在层次   | $\lfloor \log_2 i \rfloor + 1$   |

---

### 2.6 二叉树的存储结构

#### 2.6.1 顺序存储（数组表示）

用一维数组，按完全二叉树的编号来存储：

```
编号:  1  2  3  4  5  6  7
数组: [A][B][C][D][E][F][G]

对应的树：
         A(1)
       /      \
    B(2)      C(3)
   /   \     /   \
  D(4) E(5) F(6) G(7)
```

**适合完全二叉树和满二叉树**。对于一般的二叉树，需要用特殊值（如 0 或 `'\0'`）标记空缺位置，可能造成大量浪费。

```
最坏情况：一条右斜树，高度为 h，只有 h 个结点，
但需要 2^h - 1 个数组空间！
```

#### 2.6.2 链式存储（链表表示）⭐

这是最常用的存储方式。每个结点包含三个域：

In [ ]:
struct TreeNode {
    int data;               // 数据域
    TreeNode* left;         // 左子结点指针
    TreeNode* right;        // 右子结点指针
};

```
内存中的示意：

  ┌──────────────────┐
  │ data: A          │
  │ left: ──────────────→ [B结点]
  │ right: ─────────────→ [C结点]
  └──────────────────┘
```

**空指针的数量**：$n$ 个结点的二叉树有 $n + 1$ 个空指针。

**推导**：$n$ 个结点共 $2n$ 个指针域，其中有 $n - 1$ 个被使用（指向非根结点），所以空指针 = $2n - (n-1) = n + 1$。

> 💡 这些空指针后面可以用来构建**线索二叉树**（Threaded Binary Tree），提高遍历效率。

如果需要频繁访问父结点，可以增加一个 `parent` 指针：

In [ ]:
struct TreeNode {
    int data;
    TreeNode* left;
    TreeNode* right;
    TreeNode* parent;       // 三叉链表
};

---

## 第三章 二叉搜索树 (Binary Search Tree, BST)

### 3.1 什么是 BST？

二叉搜索树（也叫二叉排序树、二叉查找树）是一棵特殊的二叉树，满足：

> 对于树中的**每一个结点**：
> - 其**左子树**中所有结点的值都**小于**该结点的值
> - 其**右子树**中所有结点的值都**大于**该结点的值
> - 左右子树也分别是 BST

```
        15
       /  \
      10   20
     / \   / \
    8  12 17  25

这是一棵合法的 BST：
- 15 的左子树 {8,10,12} 全部 < 15  ✓
- 15 的右子树 {17,20,25} 全部 > 15  ✓
- 递归地，10 的左子树 {8} < 10 ✓，右子树 {12} > 10 ✓
- ... 以此类推
```

**BST 的核心价值**：查找效率高。在平衡的情况下，查找、插入、删除操作的时间复杂度为 $O(\log n)$——堪比二分查找，但比有序数组更灵活（插入删除不需要移动元素）。

### 3.2 BST 的查找操作

```
查找 12 的过程：

        15       ← 12 < 15，往左
       /
      10         ← 12 > 10，往右
        \
        12       ← 找到了！
```

In [ ]:
// 递归版本

In [ ]:
TreeNode* search(TreeNode* root, int key) {
    if (root == nullptr || root->data == key)
        return root;
    
    if (key < root->data)
        return search(root->left, key);    // 去左子树找
    else
        return search(root->right, key);   // 去右子树找
}

// 迭代版本（推荐，不会栈溢出）

In [ ]:
TreeNode* searchIterative(TreeNode* root, int key) {
    while (root != nullptr && root->data != key) {
        if (key < root->data)
            root = root->left;
        else
            root = root->right;
    }
    return root;  // 找到返回结点指针，没找到返回 nullptr
}

**时间复杂度**：
- 最好：$O(1)$（根结点就是目标）
- 平衡/平均：$O(\log n)$
- 最坏：$O(n)$（退化为链表时，例如按升序插入 1,2,3,4,5）

---

### 3.3 BST 的完整 C++ 实现

下面我们从零开始，手写一个完整的 BST。**每一行代码都有详细注释。**

#### 3.3.1 结点定义与树的创建

In [ ]:
#include <algorithm>  // 用于 max
#include <iostream>
#include <queue>      // 用于层序遍历
using namespace std;

// ============== 结点定义 ==============

In [ ]:
struct BstNode {
    int data;
    BstNode* left;
    BstNode* right;
};

In [ ]:
// 创建新结点（在堆上分配内存）
BstNode* createNode(int data) {
    BstNode* newNode = new BstNode();  // 堆上分配
    newNode->data = data;
    newNode->left = nullptr;
    newNode->right = nullptr;
    return newNode;
}

> 💡 **栈 vs 堆的内存分配**
>
> ```cpp
> // 自动存储期的局部结点：只有把它的地址带出函数才会悬空
> BstNode localNode;
> 
> // 动态分配：可让结点生命周期独立于当前函数
> BstNode* dynamicNode = new BstNode();  // ✓ 直到 delete 才释放
> ```
>
> 树的结点并非必须在堆上分配；关键是结点的生命周期必须覆盖所有使用它的指针。本例用动态分配是为了让函数返回后结点仍存活，并应最终释放；也可用对象成员、容器、智能指针或内存池管理。局部结点只有在其地址逃逸到函数外时才会形成**悬空指针 (Dangling Pointer)**。

#### 3.3.2 插入操作

In [ ]:
// 向 BST 中插入一个值

In [ ]:
BstNode* insert(BstNode* root, int data) {
    // 基本情况：找到了空位置，创建新结点
    if (root == nullptr) {
        root = createNode(data);
        return root;
    }
    
    // 递归情况：根据大小关系决定去左还是去右
    if (data < root->data) {
        root->left = insert(root->left, data);   // 插入到左子树
    } else if (data > root->data) {
        root->right = insert(root->right, data);  // 插入到右子树
    }
    // 若 data == root->data，这里选择忽略重复值
    
    return root;
}

**执行过程图解**——插入序列 `15, 10, 20, 8, 12`：

```
插入 15:         插入 10:         插入 20:         插入 8:          插入 12:
   15               15               15               15               15
                   /                /  \             /  \             /  \
                  10              10   20           10   20          10   20
                                                  /                /  \
                                                 8                8   12
```

#### 3.3.3 查找最小值和最大值

BST 的结构保证了：

- **最小值**在最左边的结点（一直往左走到底）
- **最大值**在最右边的结点（一直往右走到底）

In [ ]:
// 查找最小值

In [ ]:
int findMin(BstNode* root) {
    if (root == nullptr) {
        cout << "Error: tree is empty" << endl;
        return -1;
    }
    
    // 一直往左走，直到没有左子结点
    while (root->left != nullptr) {
        root = root->left;
    }
    return root->data;
}

// 查找最大值

In [ ]:
int findMax(BstNode* root) {
    if (root == nullptr) {
        cout << "Error: tree is empty" << endl;
        return -1;
    }
    
    // 一直往右走，直到没有右子结点
    while (root->right != nullptr) {
        root = root->right;
    }
    return root->data;
}

// 递归版本（更优雅，但实际中迭代更高效）

In [ ]:
int findMinRecursive(BstNode* root) {
    if (root->left == nullptr)
        return root->data;          // 基本情况：没有左子结点，我就是最小的
    return findMinRecursive(root->left);  // 递归：继续往左
}

**时间复杂度**：$O(h)$，$h$ 为树的高度。

---

### 3.4 二叉树的高度

In [ ]:
// 求二叉树的高度（边数定义）
// 空树高度为 -1，只有根结点的树高度为 0

In [ ]:
int findHeight(BstNode* root) {
    if (root == nullptr)
        return -1;   // 空树高度为 -1
    
    int leftHeight = findHeight(root->left);    // 左子树高度
    int rightHeight = findHeight(root->right);  // 右子树高度
    
    return max(leftHeight, rightHeight) + 1;    // 取较大值 + 1
}

**执行过程追踪**：

```
        15
       /  \
      10   20
     / \
    8  12

findHeight(15):
  leftH  = findHeight(10)
    leftH  = findHeight(8)
      leftH  = findHeight(null) = -1
      rightH = findHeight(null) = -1
      return max(-1, -1) + 1 = 0
    rightH = findHeight(12)
      return 0                     （同理）
    return max(0, 0) + 1 = 1
  rightH = findHeight(20)
    return 0                        （叶子结点）
  return max(1, 0) + 1 = 2        ← 最终结果：高度为 2
```

> ⚠️ **高度定义注意**
> - 如果教材定义空树高度为 0，只有根的树高度为 1，则基本情况改为：
> ```cpp
> if (root == nullptr) return 0;
> ```

---

## 第四章 二叉树的遍历

这是二叉树最核心的操作，没有之一。几乎所有二叉树的算法都是某种形式的遍历。

### 4.1 两大遍历策略

```
                 1
               /   \
              2     3
             / \   / \
            4   5 6   7
```

| 策略               | 英文          | 核心思想                           | 使用的数据结构           |
| ------------------ | ------------- | ---------------------------------- | ------------------------ |
| **广度优先 (BFS)** | Breadth-First | 逐层扫描，先访问完一层再访问下一层 | **队列**                 |
| **深度优先 (DFS)** | Depth-First   | 先尽可能深入到底，再回溯           | **栈**（递归本质也是栈） |

---

### 4.2 广度优先：层序遍历 (Level Order Traversal)

#### 思路

按层，从上到下、从左到右，逐个访问结点。借助**队列**实现：

1. 将根结点入队
2. 循环：出队一个结点，访问它，然后将它的左右子结点（如果存在）入队
3. 队列为空时结束

#### 执行过程

```
初始：队列 = [1]

步骤1：出队 1，访问 → 输出 1。入队 2, 3
       队列 = [2, 3]

步骤2：出队 2，访问 → 输出 2。入队 4, 5
       队列 = [3, 4, 5]

步骤3：出队 3，访问 → 输出 3。入队 6, 7
       队列 = [4, 5, 6, 7]

步骤4：出队 4，访问 → 输出 4。无子结点
步骤5：出队 5，访问 → 输出 5。无子结点
步骤6：出队 6，访问 → 输出 6。无子结点
步骤7：出队 7，访问 → 输出 7。无子结点

队列为空，结束。
输出：1 2 3 4 5 6 7
```

#### 代码实现

In [ ]:
void levelOrderTraversal(BstNode* root) {
    if (root == nullptr) return;
    
    queue<BstNode*> q;
    q.push(root);          // 根结点入队
    
    while (!q.empty()) {
        BstNode* current = q.front();  // 取队首
        q.pop();                        // 出队
        
        cout << current->data << " ";   // 访问（这里是打印）
        
        // 左子结点入队
        if (current->left != nullptr)
            q.push(current->left);
        
        // 右子结点入队
        if (current->right != nullptr)
            q.push(current->right);
    }
    cout << endl;
}

**时间复杂度**：$O(n)$，每个结点入队出队各一次。  
**空间复杂度**：$O(w)$，$w$ 为树的最大宽度。最坏情况 $O(n)$（完全二叉树的最后一层约 $n/2$ 个结点）。

---

### 4.3 深度优先：前序、中序、后序遍历

三种遍历方式的区别仅在于**访问根结点的时机**：

| 遍历方式             | 访问顺序         | 记忆口诀   |
| -------------------- | ---------------- | ---------- |
| **前序 (Preorder)**  | **根** → 左 → 右 | 根在**前** |
| **中序 (Inorder)**   | 左 → **根** → 右 | 根在**中** |
| **后序 (Postorder)** | 左 → 右 → **根** | 根在**后** |

> 💡 注意：这里的"左→右"指的是先递归处理左子树，再递归处理右子树。

#### 用例子理解

```
        1
       / \
      2   3
     / \   \
    4   5   6
```

**前序遍历**（根-左-右）：

```
访问 1
  进入左子树：访问 2
    进入左子树：访问 4
      左右子树均空，返回
    进入右子树：访问 5
      左右子树均空，返回
  进入右子树：访问 3
    左子树空
    进入右子树：访问 6

结果：1 2 4 5 3 6
```

**中序遍历**（左-根-右）：

```
  进入 1 的左子树
    进入 2 的左子树
      进入 4 的左子树（空，返回）
      访问 4
      进入 4 的右子树（空，返回）
    访问 2
    进入 2 的右子树
      访问 5
  访问 1
  进入 1 的右子树
    进入 3 的左子树（空，返回）
    访问 3
    进入 3 的右子树
      访问 6

结果：4 2 5 1 3 6
```

**后序遍历**（左-右-根）：

```
结果：4 5 2 6 3 1
```

> 🌟 **BST 的中序遍历会得到递增有序序列！**  
> 这是 BST 最重要的性质之一，很多算法都利用了这一点。

#### 递归代码实现

In [ ]:
// 前序遍历：根 → 左 → 右

In [ ]:
void preorder(BstNode* root) {
    if (root == nullptr) return;    // 基本情况
    
    cout << root->data << " ";     // 访问根
    preorder(root->left);          // 遍历左子树
    preorder(root->right);         // 遍历右子树
}

// 中序遍历：左 → 根 → 右

In [ ]:
void inorder(BstNode* root) {
    if (root == nullptr) return;
    
    inorder(root->left);           // 遍历左子树
    cout << root->data << " ";     // 访问根
    inorder(root->right);          // 遍历右子树
}

// 后序遍历：左 → 右 → 根

In [ ]:
void postorder(BstNode* root) {
    if (root == nullptr) return;
    
    postorder(root->left);         // 遍历左子树
    postorder(root->right);        // 遍历右子树
    cout << root->data << " ";     // 访问根
}

**三种遍历的代码几乎一模一样，只是 `cout` 那行所在的位置不同。** 这就是递归的优雅之处。

**时间复杂度**：$O(n)$（每个结点访问一次）  
**空间复杂度**：$O(h)$（递归栈的深度 = 树的高度）

---

### 4.4 由遍历序列构造二叉树（考试重点）

#### 核心规则

> 仅凭一种遍历序列**无法**唯一确定一棵二叉树。
> 在结点值互不相同，且两序列长度相同、包含同一组结点并确实来自某棵二叉树的前提下，以下两种组合可以**唯一确定**一棵二叉树：
> 1. **前序 + 中序**
> 2. **后序 + 中序**
>
> ⚠️ **前序 + 后序**不能唯一确定（除非是满二叉树）！

#### 方法：前序 + 中序 → 构造二叉树

**核心思路**：

1. 前序序列的**第一个元素**就是根
2. 在中序序列中找到根，根的**左边就是左子树**，**右边就是右子树**
3. 递归处理左右子树

**详细例题**：

```
前序：A B D E H C F G
中序：D B H E A F C G
```

**第 1 步**：前序第一个是 `A` → A 是根

在中序中找 A：
```
中序：[D B H E] A [F C G]
       └─左子树─┘   └─右子树─┘
```
左子树有 4 个结点，右子树有 3 个结点。

**第 2 步**：处理左子树

从前序中取出对应的 4 个元素：`B D E H`
前序子序列：`B D E H`，中序子序列：`D B H E`

前序第一个是 `B` → B 是左子树的根

在中序中找 B：
```
中序：[D] B [H E]
      └左┘   └右┘
```

**第 3 步**：继续递归...

- B 的左子树：前序 `D`，中序 `D` → 叶子结点 D
- B 的右子树：前序 `E H`，中序 `H E` → E 是根，H 是 E 的左子结点

**第 4 步**：处理右子树（A 的右子树）

前序子序列：`C F G`，中序子序列：`F C G`

前序第一个是 `C` → C 是右子树的根
在中序中找 C：
```
中序：[F] C [G]
```

F 是 C 的左子结点，G 是 C 的右子结点。

**最终结果**：

```
           A
          / \
         B   C
        / \ / \
       D  E F  G
          /
         H
```

#### 方法：后序 + 中序 → 构造二叉树

与前序+中序类似，只是**后序的最后一个元素**是根。

```
后序：D H E B F G C A
中序：D B H E A F C G

后序最后一个是 A → 根是 A
在中序中定位 A → 同样划分左右子树
递归处理...
```

#### C++ 实现

In [ ]:
#include <vector>
#include <unordered_map>

// 从前序+中序构造二叉树
// preorder: 前序序列
// inorder:  中序序列
// preStart, preEnd: 前序序列的起止索引
// inStart, inEnd:   中序序列的起止索引
// inMap: 中序序列中元素到索引的映射（加速查找）

In [ ]:
BstNode* buildTree(vector<int>& preorder, vector<int>& inorder,
                    int preStart, int preEnd,
                    int inStart, int inEnd,
                    unordered_map<int, int>& inMap) {
    if (preStart > preEnd || inStart > inEnd)
        return nullptr;
    
    // 前序的第一个元素是根
    int rootVal = preorder[preStart];
    BstNode* root = createNode(rootVal);
    
    // 在中序中找到根的位置
    int rootIndexInorder = inMap[rootVal];
    
    // 左子树的结点个数
    int leftSize = rootIndexInorder - inStart;
    
    // 递归构造左子树
    root->left = buildTree(preorder, inorder,
                           preStart + 1, preStart + leftSize,
                           inStart, rootIndexInorder - 1,
                           inMap);
    
    // 递归构造右子树
    root->right = buildTree(preorder, inorder,
                            preStart + leftSize + 1, preEnd,
                            rootIndexInorder + 1, inEnd,
                            inMap);
    
    return root;
}

// 封装函数

In [ ]:
BstNode* buildTreeFromPreIn(vector<int>& preorder, vector<int>& inorder) {
    unordered_map<int, int> inMap;
    for (int i = 0; i < inorder.size(); i++) {
        inMap[inorder[i]] = i;  // 建立中序值→索引的映射
    }
    return buildTree(preorder, inorder,
                     0, preorder.size() - 1,
                     0, inorder.size() - 1,
                     inMap);
}

---

## 第五章 BST 的高级操作

### 5.1 判断一棵二叉树是否为 BST

#### 错误方法（常见陷阱）

```
❌ 只检查每个结点的左子结点 < 根 < 右子结点

反例：
        7
       / \
      4   9
     / \
    1   8    ← 8 < 它的父结点吗？没问题。
              但 8 > 7（根结点）！违反 BST 定义！
```

仅检查"直接父子关系"是不够的，需要确保**左子树中所有结点**都小于根，**右子树中所有结点**都大于根。

#### 正确方法 1：利用范围约束

每个结点的值都有一个合法范围 `(min, max)`。向左走时更新上界，向右走时更新下界。

In [ ]:
#include <climits>

In [ ]:
bool isBstUtil(BstNode* root, long long minVal, long long maxVal) {
    if (root == nullptr)
        return true;  // 空树是 BST
    
    // 当前结点的值必须在合法范围内
    if (root->data <= minVal || root->data >= maxVal)
        return false;
    
    // 递归检查：
    // 左子树的所有值必须 < root->data
    // 右子树的所有值必须 > root->data
    return isBstUtil(root->left, minVal, root->data) &&
           isBstUtil(root->right, root->data, maxVal);
}

In [ ]:
bool isBst(BstNode* root) {
    return isBstUtil(root, LLONG_MIN, LLONG_MAX);
}

#### 正确方法 2：利用中序遍历的有序性

BST 的中序遍历结果一定是**严格递增**的。只需做一次中序遍历，检查是否递增即可。

In [ ]:
bool isBstInorder(BstNode* root, long long& prev) {
    if (root == nullptr) return true;
    
    // 先检查左子树
    if (!isBstInorder(root->left, prev))
        return false;
    
    // 检查当前结点：必须大于前一个值
    if (root->data <= prev)
        return false;
    prev = root->data;   // 更新 prev
    
    // 检查右子树
    return isBstInorder(root->right, prev);
}

In [ ]:
bool isBst2(BstNode* root) {
    long long prev = LLONG_MIN;
    return isBstInorder(root, prev);
}

---

### 5.2 BST 中删除结点（难点！）

删除是 BST 中最复杂的操作，分三种情况。

#### Case 1：删除叶子结点（度为 0）

直接删除，将父结点对应的指针置为 `nullptr`。

```
删除 2：
    5          5
   / \   →   / \
  3   7     3   7
 /
2             (直接删)
```

#### Case 2：删除只有一个子结点的结点（度为 1）

用子结点替代被删除的结点。

```
删除 3（只有左子结点）：
    5          5
   / \   →   / \
  3   7     2   7
 /
2             (用 2 替代 3)
```

#### Case 3：删除有两个子结点的结点（度为 2）⭐

这是最复杂的情况。有两种等价策略：

- **方法 A**：用**左子树的最大值**（中序前驱）替代
- **方法 B**：用**右子树的最小值**（中序后继）替代

然后递归删除那个替代结点。

```
删除 15（两个子结点）：

方法 B（用右子树最小值替代）：
        15                    17
       /  \                  /  \
      10   20       →      10   20
     / \   / \             / \     \
    8  12 17  25          8  12    25

步骤：
1. 找到右子树的最小值：17
2. 将 15 替换为 17
3. 删除右子树中的 17（此时 17 最多只有一个右子结点，属于 Case 1 或 Case 2）
```

#### 完整代码

In [ ]:
// 找到以 root 为根的子树中的最小结点

In [ ]:
BstNode* findMinNode(BstNode* root) {
    while (root->left != nullptr)
        root = root->left;
    return root;
}

// 删除 BST 中值为 data 的结点，返回删除后的根结点

In [ ]:
BstNode* deleteNode(BstNode* root, int data) {
    // 基本情况：树为空
    if (root == nullptr) 
        return root;
    
    // 递归查找要删除的结点
    if (data < root->data) {
        root->left = deleteNode(root->left, data);
    } 
    else if (data > root->data) {
        root->right = deleteNode(root->right, data);
    } 
    else {
        // 找到了要删除的结点！
        
        // Case 1 & 2：没有子结点 或 只有一个子结点
        if (root->left == nullptr) {
            BstNode* temp = root->right;
            delete root;       // 释放内存
            return temp;       // 返回右子结点（可能为 nullptr）
        }
        else if (root->right == nullptr) {
            BstNode* temp = root->left;
            delete root;
            return temp;
        }
        
        // Case 3：有两个子结点
        // 找到右子树的最小值（中序后继）
        BstNode* successor = findMinNode(root->right);
        
        // 用后继的值替换当前结点
        root->data = successor->data;
        
        // 递归删除右子树中的后继结点
        root->right = deleteNode(root->right, successor->data);
    }
    
    return root;
}

**时间复杂度**：$O(h)$，$h$ 为树高。

---

### 5.3 中序后继 (Inorder Successor)

> **中序后继**：在中序遍历序列中，某结点的下一个结点。

```
BST:
        20
       /  \
      8    22
     / \
    4   12
       / \
      10  14

中序遍历：4, 8, 10, 12, 14, 20, 22

8 的中序后继是 10
12 的中序后继是 14
14 的中序后继是 20
22 没有中序后继
```

#### 分两种情况

**Case 1**：结点有**右子树** → 后继是右子树的最小值

```
8 有右子树（以 12 为根），最小值是 10 → 后继是 10
```

**Case 2**：结点**没有右子树** → 后继是"最近的、还没有被访问过的祖先"

更准确地说：从根开始向下搜索该结点，每次**向左拐**时记录当前结点。最后一次记录的就是后继。

```
14 没有右子树。
从根 20 开始找 14：
  20 → 往左（记录 20 为候选后继）
  8 → 往右（不记录）
  12 → 往右（不记录）
  14 → 找到了

最后记录的候选后继是 20 → 14 的中序后继是 20
```

#### 代码实现

In [ ]:
BstNode* getSuccessor(BstNode* root, int data) {
    // 先找到目标结点
    BstNode* current = root;
    while (current != nullptr && current->data != data) {
        if (data < current->data)
            current = current->left;
        else
            current = current->right;
    }
    if (current == nullptr) return nullptr;
    
    // Case 1：有右子树 → 右子树的最小值
    if (current->right != nullptr) {
        return findMinNode(current->right);
    }
    
    // Case 2：没有右子树 → 从根往下找
    BstNode* successor = nullptr;
    BstNode* ancestor = root;
    
    while (ancestor != current) {
        if (current->data < ancestor->data) {
            successor = ancestor;    // 向左走时记录
            ancestor = ancestor->left;
        } else {
            ancestor = ancestor->right;  // 向右走时不记录
        }
    }
    
    return successor;
}

---

## 第六章 树的存储结构

前面我们重点讲了二叉树。现在回到**一般的树**（结点可以有任意多个子结点）。

### 6.1 双亲表示法（数组）

用一维数组存储所有结点，每个结点记录其**父结点的下标**。

In [ ]:
#define MAX_SIZE 100

In [ ]:
struct PTNode {
    char data;      // 数据
    int parent;     // 父结点在数组中的下标（-1 表示根）
};

In [ ]:
struct PTree {
    PTNode nodes[MAX_SIZE];
    int n;          // 结点总数
};

```
树：        A
           /|\
          B C D
         / \
        E   F

数组表示：
下标  data  parent
 0     A     -1     ← 根
 1     B      0
 2     C      0
 3     D      0
 4     E      1
 5     F      1
```

**优点**：找父结点 $O(1)$。  
**缺点**：找子结点需要遍历整个数组 $O(n)$。

### 6.2 孩子表示法（邻接表）

每个结点维护一个**链表**，链表中存储所有子结点。

```
A → [B] → [C] → [D] → null
B → [E] → [F] → null
C → null
D → null
E → null
F → null
```

**优点**：找子结点方便。  
**缺点**：找父结点不方便。

### 6.3 孩子兄弟表示法（二叉链表）⭐

**最重要的存储方式！** 它能将任意树转换为二叉树。

每个结点有两个指针：
- `firstChild`：指向**第一个子结点**
- `nextSibling`：指向**下一个兄弟结点**

In [ ]:
struct CSNode {
    char data;
    CSNode* firstChild;    // 左指针：第一个子结点
    CSNode* nextSibling;   // 右指针：右兄弟
};

```
原树：          A
              / | \
             B  C  D
            / \
           E   F

孩子兄弟表示法：
         A
        /
       B → C → D
      /
     E → F

这其实就是一棵二叉树！
         A
        /
       B
      / \
     E   C
      \   \
       F   D
```

---

## 第七章 树、森林与二叉树的转换

### 7.1 树 → 二叉树

**规则**（口诀：**左孩子右兄弟**）：

1. 保留每个结点与**第一个子结点**的连线（作为左子结点）
2. 在兄弟结点之间加连线（作为右子结点）
3. 删除原来除第一个子结点外的所有连线

**详细步骤图解**：

```
原树：             A
                /  |  \
               B   C   D
              / \      |
             E   F     G

Step 1：加兄弟连线
               A
              /|  \
             B-C---D
            /\     |
           E--F    G

Step 2：只保留第一个子结点连线，删除其余父子连线
               A
              /
             B---C---D
            /        |
           E---F     G

Step 3：顺时针旋转 45°，得到二叉树
                A
               /
              B
             / \
            E   C
             \   \
              F   D
                 /
                G
```

> 🌟 **重要结论**：树转换成的二叉树，**根结点一定没有右子树**（因为根没有兄弟）。

### 7.2 森林 → 二叉树

1. 先将每棵树各自转换为二叉树
2. 将第 2 棵二叉树作为第 1 棵二叉树根的**右子树**
3. 将第 3 棵二叉树作为第 2 棵二叉树根的**右子树**
4. 依次类推……

```
森林：
  T1:  A        T2:  D        T3:  G
      / \           / \
     B   C         E   F

各自转换为二叉树：
  T1':    A      T2':    D      T3':  G
         /              /
        B              E
         \              \
          C              F

串联（右子树链）：
           A
          / \
         B   D
          \ / \
          C E   G
             \
              F
```

### 7.3 二叉树 → 树/森林

上述过程的逆操作：

1. 如果某结点是父结点的**左子结点**，则它是父结点的子结点
2. 沿着右子结点链，所有结点都是**同一层的兄弟**
3. 恢复父子连线，删除兄弟连线

**判断标准**：
- 二叉树的根**没有右子树** → 还原为**一棵树**
- 二叉树的根**有右子树** → 还原为**森林**

---

## 第八章 树和森林的遍历

### 8.1 树的遍历

| 遍历方式     | 访问顺序                       | 等价于对应二叉树的             |
| ------------ | ------------------------------ | ------------------------------ |
| **先根遍历** | 先访问根，再依次先根遍历各子树 | **前序遍历**                   |
| **后根遍历** | 先依次后根遍历各子树，再访问根 | **中序遍历**                   |
| **层次遍历** | 逐层从左到右                   | （用队列，类似二叉树层序遍历） |

**例子**：

```
树：          A
            / | \
           B  C  D
          / \    |
         E   F   G
```

- **先根遍历**：A B E F C D G
- **后根遍历**：E F B C G D A

**验证**：转换为二叉树后

```
二叉树：      A
             /
            B
           / \
          E   C
           \   \
            F   D
               /
              G
```

- **前序遍历**：A B E F C D G ✓（等于先根遍历）
- **中序遍历**：E F B C G D A ✓（等于后根遍历）

### 8.2 森林的遍历

| 遍历方式     | 访问顺序                                                     | 等价于对应二叉树的 |
| ------------ | ------------------------------------------------------------ | ------------------ |
| **先序遍历** | 访问第一棵树的根 → 先序遍历第一棵树的子树森林 → 先序遍历剩余树的森林 | **前序遍历**       |
| **中序遍历** | 中序遍历第一棵树的子树森林 → 访问第一棵树的根 → 中序遍历剩余树的森林 | **中序遍历**       |

---

## 第九章 哈夫曼树 (Huffman Tree)

### 9.1 基本概念

#### 9.1.1 带权路径长度 (WPL)

> **结点的权 (Weight)**：赋予结点的一个数值，代表某种"重要程度"或"频率"。
>
> **结点的带权路径长度**：从根到该结点的路径长度 × 该结点的权值。
>
> **树的带权路径长度 (Weighted Path Length)**：所有**叶子结点**的带权路径长度之和。

$$WPL = \sum_{i=1}^{n} w_i \cdot l_i$$

其中 $w_i$ 是第 $i$ 个叶子的权值，$l_i$ 是根到第 $i$ 个叶子的路径长度。

**例子**：

```
权值集合：{2, 3, 5, 7}

树 A：                树 B：
      ○                   ○
    /   \               / | \
   ○     ○             2  3  ○
  / \   / \                  / \
 2   3 5   7                5   7

WPL_A = 2×2 + 3×2 + 5×2 + 7×2 = 34
WPL_B = 2×1 + 3×1 + 5×2 + 7×2 = 29   （注意：这不是二叉树，但说明 WPL 概念）
```

下面构造权值 {2, 3, 5, 7} 的哈夫曼树。

### 9.2 哈夫曼树的构造算法

#### 核心思想（贪心策略）

> 每次选择**权值最小的两个结点**合并，合并后的新结点的权值等于两者之和。重复此过程直到只剩一棵树。

#### 详细构造过程

**初始权值**：{2, 3, 5, 7}

```
Step 1：选最小的两个：2 和 3，合并
         5'         （新结点，权值 = 2+3 = 5）
        / \
       2   3

当前待合并集合：{5', 5, 7}   （5' 是新结点，5 是原来的）

Step 2：选最小的两个：5' 和 5（两个都是 5，选哪个都行），合并
         10
        /  \
       5'   5
      / \
     2   3

当前待合并集合：{10, 7}

Step 3：选最小的两个：7 和 10，合并
           17
          /  \
         7   10
            /  \
           5'   5
          / \
         2   3

这就是哈夫曼树！
```

**计算 WPL**：
```
WPL = 7×1 + 5×2 + 2×3 + 3×3 = 7 + 10 + 6 + 9 = 32
```

**这是所有可能的二叉树中 WPL 最小的！** 所以哈夫曼树又叫**最优二叉树**。

#### 哈夫曼树的重要性质

1. **权值越大的叶子越靠近根**（路径越短）
2. **不存在度为 1 的结点**（每次合并两个，所以只有度 0 和度 2 的结点）
3. **$n$ 个叶子结点的哈夫曼树共有 $2n - 1$ 个结点**
   - 推导：$n_0 = n_2 + 1$（二叉树性质），又 $n_1 = 0$，所以 $n_2 = n - 1$，总结点 = $n + (n-1) = 2n - 1$
4. **哈夫曼树不唯一**，但 WPL 一定是最小的

### 9.3 哈夫曼编码 (Huffman Coding)

#### 背景：为什么需要哈夫曼编码？

假设有一段文本包含字符 A, B, C, D，出现频率分别为：

| 字符 | 频率 |
| ---- | ---- |
| A    | 45%  |
| B    | 13%  |
| C    | 12%  |
| D    | 16%  |
| E    | 9%   |
| F    | 5%   |

如果只编码这里的 6 种字符，等长二进制编码至少需要 3 位（这不是 ASCII）：
```
A=000, B=001, C=010, D=011, E=100, F=101
```
100 个字符需要 300 位。

**能否根据频率高低，给高频字符更短的编码？**

#### 前缀编码的要求

编码必须是**前缀编码**：任何一个字符的编码都不是另一个字符编码的前缀。

```
❌ 非前缀码示例：A=0, B=01, C=1
   → 收到 "01" 时，可解为 B(01) 或 A C(0|1)

✓ 正确编码：A=0, B=10, C=110, D=111
   → 任何编码都不是其他编码的前缀，可以唯一解码
```

#### 构造哈夫曼编码

1. 以字符频率作为权值，构造哈夫曼树
2. 左分支标记 `0`，右分支标记 `1`
3. 从根到叶子的路径即为该字符的编码

**例子**：频率 A=45, B=13, C=12, D=16, E=9, F=5

```
Step 1：合并 F(5) 和 E(9) → 14
Step 2：合并 C(12) 和 B(13) → 25
Step 3：合并 14 和 D(16) → 30
Step 4：合并 25 和 30 → 55
Step 5：合并 A(45) 和 55 → 100

哈夫曼树：
                  100
                /     \
             A(45)    55
                     /   \
                   25     30
                  / \    /  \
               C(12) B(13) 14  D(16)
                          / \
                        F(5) E(9)
```

标记路径（左 0，右 1）：

```
                  100
                0/   \1
             A(45)    55
                    0/   \1
                   25     30
                 0/ \1  0/  \1
              C(12) B(13) 14  D(16)
                        0/ \1
                       F(5) E(9)
```

| 字符 | 编码 | 长度 |
| ---- | ---- | ---- |
| A    | 0    | 1 位 |
| C    | 100  | 3 位 |
| B    | 101  | 3 位 |
| F    | 1100 | 4 位 |
| E    | 1101 | 4 位 |
| D    | 111  | 3 位 |

**加权平均编码长度** = $0.45×1 + 0.13×3 + 0.12×3 + 0.16×3 + 0.09×4 + 0.05×4 = 0.45 + 0.39 + 0.36 + 0.48 + 0.36 + 0.20 = 2.24$ 位/字符

比等长编码的 3 位/字符节省了 25% 的空间！

### 9.4 哈夫曼树的 C++ 实现

In [ ]:
#include <iostream>
#include <queue>
#include <vector>
#include <string>
#include <unordered_map>
using namespace std;

// 哈夫曼树结点

In [ ]:
struct HuffNode {
    char ch;        // 字符（只有叶子结点才有意义）
    int freq;       // 频率/权值
    HuffNode* left;
    HuffNode* right;
    
    HuffNode(char c, int f) : ch(c), freq(f), left(nullptr), right(nullptr) {}
};

In [ ]:
// 自定义比较器：小顶堆（频率小的优先）
struct Compare {
    bool operator()(HuffNode* a, HuffNode* b) {
        return a->freq > b->freq;  // 注意：> 号，因为 priority_queue 默认大顶堆
    }
};

In [ ]:
// 构造哈夫曼树
HuffNode* buildHuffmanTree(unordered_map<char, int>& freqMap) {
    if (freqMap.empty()) return nullptr;

    // 用优先队列（最小堆）实现"每次取最小的两个"
    priority_queue<HuffNode*, vector<HuffNode*>, Compare> minHeap;
    
    // 将所有字符作为叶子结点加入堆
    for (auto& pair : freqMap) {
        minHeap.push(new HuffNode(pair.first, pair.second));
    }
    
    // 反复合并，直到堆中只剩一个结点
    while (minHeap.size() > 1) {
        HuffNode* left = minHeap.top(); minHeap.pop();   // 取最小
        HuffNode* right = minHeap.top(); minHeap.pop();  // 取次小
        
        // 创建新的内部结点
        HuffNode* parent = new HuffNode('\0', left->freq + right->freq);
        parent->left = left;
        parent->right = right;
        
        minHeap.push(parent);  // 放回堆中
    }
    
    return minHeap.top();  // 堆中最后一个就是根
}

// 生成哈夫曼编码（递归遍历）

In [ ]:
void generateCodes(HuffNode* root, string code, 
                   unordered_map<char, string>& huffCodes) {
    if (root == nullptr) return;
    
    // 叶子结点：记录编码
    if (root->left == nullptr && root->right == nullptr) {
        huffCodes[root->ch] = code.empty() ? "0" : code;
        return;
    }
    
    generateCodes(root->left, code + "0", huffCodes);   // 左走加 0
    generateCodes(root->right, code + "1", huffCodes);  // 右走加 1
}

// 编码：将字符串转换为哈夫曼编码

In [ ]:
string encode(const string& text, unordered_map<char, string>& huffCodes) {
    string encoded = "";
    for (char c : text) {
        encoded += huffCodes[c];
    }
    return encoded;
}

// 解码：将编码还原为字符串

In [ ]:
string decode(const string& encoded, HuffNode* root) {
    string decoded = "";
    if (root == nullptr) return decoded;

    // 单字符树没有边；约定唯一字符的编码为 0。
    if (root->left == nullptr && root->right == nullptr) {
        for (char bit : encoded) {
            if (bit == '0') decoded += root->ch;
        }
        return decoded;
    }

    HuffNode* current = root;
    
    for (char bit : encoded) {
        if (bit == '0')
            current = current->left;
        else
            current = current->right;
        
        // 到达叶子结点：输出字符，回到根
        if (current->left == nullptr && current->right == nullptr) {
            decoded += current->ch;
            current = root;
        }
    }
    
    return decoded;
}

// 测试

In [ ]:
int main() {
    unordered_map<char, int> freqMap = {
        {'A', 45}, {'B', 13}, {'C', 12}, 
        {'D', 16}, {'E', 9},  {'F', 5}
    };
    
    HuffNode* root = buildHuffmanTree(freqMap);
    
    unordered_map<char, string> huffCodes;
    generateCodes(root, "", huffCodes);
    
    cout << "=== 哈夫曼编码表 ===" << endl;
    for (auto& pair : huffCodes) {
        cout << pair.first << " : " << pair.second << endl;
    }
    
    string text = "ABCDEF";
    string encoded = encode(text, huffCodes);
    cout << "\n原文: " << text << endl;
    cout << "编码: " << encoded << endl;
    
    string decoded = decode(encoded, root);
    cout << "解码: " << decoded << endl;
    
    return 0;
}

In [ ]:
main();

---

## 第十章 完整 BST 综合程序

最后，我们将前面学过的所有 BST 操作整合成一个完整可运行的程序：

In [ ]:
#include <algorithm>
#include <climits>
#include <iostream>
#include <queue>
#include <string>
using namespace std;

// ================== 结点定义 ==================

In [ ]:
struct BstNode {
    int data;
    BstNode* left;
    BstNode* right;
};

In [ ]:
BstNode* createNode(int data) {
    BstNode* node = new BstNode();
    node->data = data;
    node->left = node->right = nullptr;
    return node;
}

// ================== 插入 ==================

In [ ]:
BstNode* insert(BstNode* root, int data) {
    if (root == nullptr) return createNode(data);
    if (data < root->data)
        root->left = insert(root->left, data);
    else if (data > root->data)
        root->right = insert(root->right, data);
    return root;
}

// ================== 查找 ==================

In [ ]:
bool search(BstNode* root, int data) {
    if (root == nullptr) return false;
    if (data == root->data) return true;
    if (data < root->data) return search(root->left, data);
    return search(root->right, data);
}

// ================== 最小值/最大值 ==================

In [ ]:
BstNode* findMinNode(BstNode* root) {
    while (root && root->left) root = root->left;
    return root;
}

In [ ]:
int findMax(BstNode* root) {
    while (root && root->right) root = root->right;
    return root ? root->data : -1;
}

// ================== 高度 ==================

In [ ]:
int height(BstNode* root) {
    if (root == nullptr) return -1;
    return max(height(root->left), height(root->right)) + 1;
}

// ================== 遍历 ==================

In [ ]:
void preorder(BstNode* root) {
    if (!root) return;
    cout << root->data << " ";
    preorder(root->left);
    preorder(root->right);
}

In [ ]:
void inorder(BstNode* root) {
    if (!root) return;
    inorder(root->left);
    cout << root->data << " ";
    inorder(root->right);
}

In [ ]:
void postorder(BstNode* root) {
    if (!root) return;
    postorder(root->left);
    postorder(root->right);
    cout << root->data << " ";
}

In [ ]:
void levelOrder(BstNode* root) {
    if (!root) return;
    queue<BstNode*> q;
    q.push(root);
    while (!q.empty()) {
        BstNode* curr = q.front(); q.pop();
        cout << curr->data << " ";
        if (curr->left) q.push(curr->left);
        if (curr->right) q.push(curr->right);
    }
}

// ================== 判断是否为 BST ==================

In [ ]:
bool isBstUtil(BstNode* root, long long minVal, long long maxVal) {
    if (root == nullptr) return true;
    if (root->data <= minVal || root->data >= maxVal) return false;
    return isBstUtil(root->left, minVal, root->data) &&
           isBstUtil(root->right, root->data, maxVal);
}

In [ ]:
bool isBst(BstNode* root) {
    return isBstUtil(root, LLONG_MIN, LLONG_MAX);
}

// ================== 删除 ==================

In [ ]:
BstNode* deleteNode(BstNode* root, int data) {
    if (root == nullptr) return nullptr;
    if (data < root->data)
        root->left = deleteNode(root->left, data);
    else if (data > root->data)
        root->right = deleteNode(root->right, data);
    else {
        if (root->left == nullptr) {
            BstNode* temp = root->right;
            delete root;
            return temp;
        } else if (root->right == nullptr) {
            BstNode* temp = root->left;
            delete root;
            return temp;
        }
        BstNode* succ = findMinNode(root->right);
        root->data = succ->data;
        root->right = deleteNode(root->right, succ->data);
    }
    return root;
}

// ================== 中序后继 ==================

In [ ]:
BstNode* getSuccessor(BstNode* root, int data) {
    BstNode* current = root;
    // 先找到目标结点
    while (current && current->data != data) {
        if (data < current->data) current = current->left;
        else current = current->right;
    }
    if (!current) return nullptr;
    
    if (current->right)
        return findMinNode(current->right);
    
    BstNode* successor = nullptr;
    BstNode* ancestor = root;
    while (ancestor != current) {
        if (current->data < ancestor->data) {
            successor = ancestor;
            ancestor = ancestor->left;
        } else {
            ancestor = ancestor->right;
        }
    }
    return successor;
}

// ================== 主函数 ==================

In [ ]:
int main() {
    BstNode* root = nullptr;
    
    // 构建 BST
    int values[] = {15, 10, 20, 8, 12, 17, 25};
    for (int v : values)
        root = insert(root, v);
    
    cout << "========= BST 操作演示 =========" << endl;
    
    // 遍历
    cout << "前序遍历: "; preorder(root); cout << endl;
    cout << "中序遍历: "; inorder(root); cout << endl;
    cout << "后序遍历: "; postorder(root); cout << endl;
    cout << "层序遍历: "; levelOrder(root); cout << endl;
    
    // 查找
    cout << "\n查找 12: " << (search(root, 12) ? "找到" : "未找到") << endl;
    cout << "查找 99: " << (search(root, 99) ? "找到" : "未找到") << endl;
    
    // 最值
    cout << "\n最小值: " << findMinNode(root)->data << endl;
    cout << "最大值: " << findMax(root) << endl;
    
    // 高度
    cout << "\n树的高度: " << height(root) << endl;
    
    // 是否为 BST
    cout << "是否为 BST: " << (isBst(root) ? "是" : "否") << endl;
    
    // 中序后继
    BstNode* succ = getSuccessor(root, 12);
    cout << "\n12 的中序后继: " << (succ ? to_string(succ->data) : "无") << endl;
    
    succ = getSuccessor(root, 15);
    cout << "15 的中序后继: " << (succ ? to_string(succ->data) : "无") << endl;
    
    // 删除
    cout << "\n删除 10 后的中序遍历: ";
    root = deleteNode(root, 10);
    inorder(root); cout << endl;
    
    cout << "删除 15 后的中序遍历: ";
    root = deleteNode(root, 15);
    inorder(root); cout << endl;
    
    return 0;
}

In [ ]:
main();

**预期输出**：

```
========= BST 操作演示 =========
前序遍历: 15 10 8 12 20 17 25
中序遍历: 8 10 12 15 17 20 25
后序遍历: 8 12 10 17 25 20 15
层序遍历: 15 10 20 8 12 17 25

查找 12: 找到
查找 99: 未找到

最小值: 8
最大值: 25

树的高度: 2
是否为 BST: 是

12 的中序后继: 15
15 的中序后继: 17

删除 10 后的中序遍历: 8 12 15 17 20 25
删除 15 后的中序遍历: 8 12 17 20 25
```

---

## 知识总结与考试速查表

### 核心公式速查

| 公式                                                        | 适用范围             |
| ----------------------------------------------------------- | -------------------- |
| $n = \text{总度数} + 1$                                     | 任意树               |
| 第 $i$ 层最多 $m^{i-1}$ 个结点                              | 度为 $m$ 的树        |
| 高度 $h$ 最多 $\dfrac{m^h - 1}{m - 1}$ 个结点               | $m$ 叉树             |
| $n_0 = n_2 + 1$                                             | 二叉树               |
| 完全二叉树高度 $h = \lfloor \log_2 n \rfloor + 1$           | 完全二叉树           |
| 结点 $i$ 的左子 $2i$，右子 $2i+1$，父 $\lfloor i/2 \rfloor$ | 完全二叉树层序编号   |
| $n$ 个结点有 $n+1$ 个空指针                                 | 二叉链表             |
| 哈夫曼树有 $2n-1$ 个结点                                    | $n$ 个叶子的哈夫曼树 |

### 遍历对应关系

| 树的遍历 | 对应二叉树的遍历 |
| -------- | ---------------- |
| 先根遍历 | 前序遍历         |
| 后根遍历 | 中序遍历         |

| 森林的遍历 | 对应二叉树的遍历 |
| ---------- | ---------------- |
| 先序遍历   | 前序遍历         |
| 中序遍历   | 中序遍历         |

### 转换规则

| 转换             | 核心规则                         |
| ---------------- | -------------------------------- |
| 树 → 二叉树      | 左孩子右兄弟                     |
| 森林 → 二叉树    | 各树转二叉树后右链串联           |
| 二叉树 → 树/森林 | 反向操作：沿右链断开恢复兄弟关系 |

### BST 操作复杂度

| 操作   | 平均        | 最坏（退化为链表） |
| ------ | ----------- | ------------------ |
| 查找   | $O(\log n)$ | $O(n)$             |
| 插入   | $O(\log n)$ | $O(n)$             |
| 删除   | $O(\log n)$ | $O(n)$             |
| 找最值 | $O(\log n)$ | $O(n)$             |

---

> 📚 **学完本章后，你应该能够**：
> 1. 清楚地说出树、二叉树、BST 的定义和性质
> 2. 手写 BST 的插入、查找、删除、遍历操作
> 3. 根据前序+中序（或后序+中序）还原一棵二叉树
> 4. 将任意树/森林与二叉树互相转换
> 5. 构造哈夫曼树并生成哈夫曼编码
> 6. 熟练运用各种性质公式解决选择题